In [1]:
import sys

print(sys.executable)

E:\CAROLA\CAROLA TRABAJO\PASANTIA UNNE - AGENTE ASUNTOS LEGALES\pasantia-segem-agentes-ia\.venv\Scripts\python.exe


In [6]:
import sys
import subprocess

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "pandas",
    "beautifulsoup4",
    "lxml",
    "html5lib",
    "ftfy",
    "unidecode",
    "markdownify",
    "tqdm"
])

0

# 01 - Exploración inicial de CSV y JSON

Objetivo: entender la estructura inicial de los archivos entregados para la pasantía.

En esta etapa vamos a responder:

- Cuántos registros hay.
- Qué columnas/campos tiene cada archivo.
- Dónde podría estar el texto legal del oficio/embargo.
- Si hay HTML dentro del texto.
- Si hay valores nulos.
- Si hay documentos repetidos.
- Si hay tipos de documento.
- Si hay respuestas generadas.

# 1. Verificar entorno y rutas

Primera celda de código:

In [9]:
import sys
from pathlib import Path

print("Python ejecutándose desde:")
print(sys.executable)

Python ejecutándose desde:
E:\CAROLA\CAROLA TRABAJO\PASANTIA UNNE - AGENTE ASUNTOS LEGALES\pasantia-segem-agentes-ia\.venv\Scripts\python.exe


In [10]:
PROJECT_ROOT = Path.cwd().parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

print("Proyecto:", PROJECT_ROOT)
print("Raw:", RAW_DIR)
print("Processed:", PROCESSED_DIR)

print("Existe data/raw:", RAW_DIR.exists())
print("Existe data/processed:", PROCESSED_DIR.exists())

Proyecto: E:\CAROLA\CAROLA TRABAJO\PASANTIA UNNE - AGENTE ASUNTOS LEGALES\pasantia-segem-agentes-ia
Raw: E:\CAROLA\CAROLA TRABAJO\PASANTIA UNNE - AGENTE ASUNTOS LEGALES\pasantia-segem-agentes-ia\data\raw
Processed: E:\CAROLA\CAROLA TRABAJO\PASANTIA UNNE - AGENTE ASUNTOS LEGALES\pasantia-segem-agentes-ia\data\processed
Existe data/raw: True
Existe data/processed: True


# 2. Imports principales

In [11]:
import pandas as pd
import json
import re
from bs4 import BeautifulSoup
from IPython.display import display

print("Librerías importadas correctamente")
print("Pandas:", pd.__version__)

Librerías importadas correctamente
Pandas: 3.0.3


# 3. Ver qué archivos hay en data/raw

In [12]:
archivos_raw = list(RAW_DIR.iterdir())

print("Archivos encontrados en data/raw:")
for archivo in archivos_raw:
    print("-", archivo.name)

Archivos encontrados en data/raw:
- Entradas origen drive.csv
- Entradas origen drive.json


# 4. Definir nombres de archivos

In [14]:
csv_path = RAW_DIR / "Entradas origen drive.csv"
json_path = RAW_DIR / "Entradas origen drive.json"

print("CSV existe:", csv_path.exists())
print("JSON existe:", json_path.exists())

CSV existe: True
JSON existe: True


# 5. Cargar CSV

In [15]:
df_csv = pd.read_csv(csv_path)

print("CSV cargado correctamente")
print("Filas y columnas:", df_csv.shape)

CSV cargado correctamente
Filas y columnas: (2986, 4)


# 6. Ver primeras filas del CSV

In [16]:
df_csv.head()
df_csv.tail()

,id,texto_ocr,nombre,clasificacion
2981,547398,Error al enviar al textify.,Solicitud de informacion de usuarios,Oficio
2982,547399,20 MAR 2026\n\ngr\nRE ERAS duniciaL REITERATOR...,Embargo - usuario,Embargo
2983,547401,Poder Judicial de la Nación\nJUZGADO CIVIL 29\...,Embargo - usuario,Embargo
2984,547402,Poder Judicial de la Nación\n\nÑ PEL\n\nCédula...,Demanda,Juicio
2985,547403,Error al descargar el archivo desde el bucket.,Demanda,Juicio


# 7. Ver columnas del CSV

In [17]:
columnas_csv = df_csv.columns.tolist()

print("Cantidad de columnas:", len(columnas_csv))
print("Columnas del CSV:")

for col in columnas_csv:
    print("-", col)

Cantidad de columnas: 4
Columnas del CSV:
- id
- texto_ocr
- nombre
- clasificacion


# 8. Información general del CSV

In [18]:
df_csv.info()

<class 'pandas.DataFrame'>
RangeIndex: 2986 entries, 0 to 2985
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   id             2986 non-null   int64
 1   texto_ocr      2986 non-null   str  
 2   nombre         2986 non-null   str  
 3   clasificacion  2986 non-null   str  
dtypes: int64(1), str(3)
memory usage: 93.4 KB


# 9. Valores nulos por columna

In [19]:
nulos_csv = df_csv.isna().sum().sort_values(ascending=False)

nulos_csv

id               0
texto_ocr        0
nombre           0
clasificacion    0
dtype: int64

## Versión más linda en tabla:

In [20]:
reporte_nulos_csv = pd.DataFrame({
    "columna": df_csv.columns,
    "nulos": df_csv.isna().sum().values,
    "porcentaje_nulos": (df_csv.isna().mean().values * 100).round(2)
}).sort_values(by="nulos", ascending=False)

display(reporte_nulos_csv)

,columna,nulos,porcentaje_nulos
0,id,0,0.0
1,texto_ocr,0,0.0
2,nombre,0,0.0
3,clasificacion,0,0.0


# 10. Detectar columnas de texto largo en CSV

In [24]:
def analizar_columnas_texto(df):
    resultados = []

    for col in df.columns:
        serie_original = df[col].dropna()

        if len(serie_original) == 0:
            continue

        # Convertimos a texto para poder medir largo, aunque pandas no la haya detectado como object
        serie = serie_original.astype(str)
        largos = serie.str.len()

        resultados.append({
            "columna": col,
            "tipo_dato": str(df[col].dtype),
            "valores_no_nulos": int(serie.count()),
            "promedio_caracteres": round(largos.mean(), 2),
            "max_caracteres": int(largos.max()),
            "min_caracteres": int(largos.min())
        })

    if len(resultados) == 0:
        print("No se encontraron columnas con datos para analizar.")
        return pd.DataFrame()

    return pd.DataFrame(resultados).sort_values(
        by="promedio_caracteres",
        ascending=False
    )

In [25]:
reporte_texto_csv = analizar_columnas_texto(df_csv)

display(reporte_texto_csv)

,columna,tipo_dato,valores_no_nulos,promedio_caracteres,max_caracteres,min_caracteres
1,texto_ocr,str,2986,3287.25,117897,1
2,nombre,str,2986,18.33,49,5
3,clasificacion,str,2986,9.46,18,5
0,id,int64,2986,6.00,6,6


# 11. Ver ejemplos de columnas candidatas
Después de ver el reporte, elegí una columna candidata.  

Por ejemplo, si aparece una columna llamada body, texto, descripcion, html, documento, content, etc.  

In [26]:
COLUMNA_TEXTO_CSV = "texto_ocr"

for i, texto in enumerate(df_csv[COLUMNA_TEXTO_CSV].dropna().astype(str).head(5), start=1):
    print("=" * 100)
    print(f"EJEMPLO CSV {i}")
    print("=" * 100)
    print(texto[:3000])
    print()

EJEMPLO CSV 1
ii
4373-350
o.gestona.profesionalU gmail.com

A A A A

Tel:

FEDERAL

MERCADO PAGI

toría Profesional
21 011 - 4373-3500

lor a profesional) gmail.com

AE isaoss s326 MXN ÍN
DO Bo

0.4
Poder Judicial de la Nación

JUZGADO CIVIL 53

OFICIO. CON FIRMA ELECTRÓNICA CONFORME
ACORDADA 12/2020 C.S.JN (para cotejo de certificado de
validación de firma, descargar documento en: consulta de causas
en www.pjn.gov.ar//consulta de causas/ jurisdicción: cámara

nacional en lo civil// número de expediente// año).

mercado
libre OFICIO
ENE 2026 EMBARGO
pi
EPCION Buenos Aires, de diciembre de 2025

Mercado Pago Servicios de procesamiento S.R.L.
Avenida Caseros nro. 3039 piso 2 de CABA.

S / D:

Tengo el agrado de dirigirme a Usted en los autos
caratulados “LA VICTORIA POLO COUNTRY CLUB S.A. e/
RAMIREZ, MATIAS s/COBRO DE SUMAS DE DINERO”
EXP. 12611/2025 en trámite por ante el Juzgado Nacional de
Primera Instancia CIVIL NRO. 53, sito en Avenida de los
Inmigrantes nro. 1950 PISO 6 de CABA a c

# 12. Detectar si hay HTML

In [27]:
def parece_html(texto):
    if pd.isna(texto):
        return False

    texto = str(texto).lower()

    patrones_html = [
        "<html", "<body", "<p", "<div", "<br", "<span", "<table",
        "<tr", "<td", "<strong", "<b", "&nbsp;", "&amp;", "</"
    ]

    return any(patron in texto for patron in patrones_html)

Aplicamos sobre la columna candidata:

In [28]:
df_csv["tiene_html_detectado"] = df_csv[COLUMNA_TEXTO_CSV].apply(parece_html)

df_csv["tiene_html_detectado"].value_counts()

tiene_html_detectado
False    2958
True       28
Name: count, dtype: int64

Y en porcentaje:

In [29]:
porcentaje_html_csv = df_csv["tiene_html_detectado"].mean() * 100

print(f"Porcentaje de registros con posible HTML: {porcentaje_html_csv:.2f}%")

Porcentaje de registros con posible HTML: 0.94%


# 13. Ver ejemplos que tienen HTML

In [30]:
ejemplos_html_csv = df_csv[df_csv["tiene_html_detectado"] == True][COLUMNA_TEXTO_CSV].dropna().astype(str).head(3)

for i, texto in enumerate(ejemplos_html_csv, start=1):
    print("=" * 100)
    print(f"EJEMPLO CON HTML {i}")
    print("=" * 100)
    print(texto[:3000])
    print()

EJEMPLO CON HTML 1
la lLoy NO 2 AROSA lo

REMITENTE 772201

/ICIOS DE PROCESAMIENTO SAL ARMANY CAROLINA PERDOMO ORTIZ
Apellido y nombre .

hb razón social q

ISA y E 96.176.316
Rama o al pal ÓN Ne
BISO 2 (1264) Ñ AV. CORRIENTES 2335, OF. 1* “A” Fecha
Código Postal Donnicilio real (1046)
Ñ C.A.B.A. C.A.B.A. Codigo Postai
Provincia Localidad TABA.

Provincia

lidano, remítole copia del requerimiento efecmado 2 CONTACTO GARANTIDO S.A, rigicodo a Ud. las mismas intimaciones y «perdbimientos:
to CD 35094083 O, de fecha 13/12/2025, por resultar sus términos fsos, maliciosos e improcedentes. Ratifico en todos sus términos mis anteriores
que reiteradamente vengo reclamando y que Ud. persiste en 1acumplix, y la cotalidad de los reclamos, jotimaciones y apercibimienzos efecnlados en

estras manifestaciones. Niego y rechazo una vez más encontrarme po reserva de puesto como Ud, refiere, en relación a lo cual le reitero —evamente
Ceabajo a disposición, la reserva de puesto ha cesado cuando mi médic

# Mejor verificación del HTML

Función para saber qué patrón activó el True:

In [31]:
def detectar_patrones_html(texto):
    if pd.isna(texto):
        return []

    texto = str(texto).lower()

    patrones_html = [
        "<html", "<body", "<p", "<div", "<br", "<span", "<table",
        "<tr", "<td", "<strong", "<b", "&nbsp;", "&amp;", "</"
    ]

    encontrados = []

    for patron in patrones_html:
        if patron in texto:
            encontrados.append(patron)

    return encontrados

Ejecutar: 

In [32]:
df_csv["patrones_html_detectados"] = df_csv[COLUMNA_TEXTO_CSV].apply(detectar_patrones_html)

df_csv[df_csv["tiene_html_detectado"] == True][
    ["id", "nombre", "clasificacion", "patrones_html_detectados"]
].head(28)

,id,nombre,clasificacion,patrones_html_detectados
175,496886,Intimacion,Juicio,[<p]
335,498324,Audiencia,Oficio,[<p]
403,499481,Audiencia,Oficio,[<p]
425,500696,Reclamo,Defensa Consumidor,[<b]
428,500699,Audiencia,Oficio,[<tr]
513,502596,Reclamo,Defensa Consumidor,[<p]
514,502597,Carta Simple,Defensa Consumidor,[<b]
582,502668,Intimacion,Juicio,[<p]
611,503769,Audiencia,Oficio,"[<p, <br, <b]"
960,510943,Audiencia,Oficio,[<p]


# Ver ejemplos más claros de esos casos

In [33]:
casos_html = df_csv[df_csv["tiene_html_detectado"] == True]

for i, row in casos_html.head(5).iterrows():
    print("=" * 100)
    print("ID:", row["id"])
    print("NOMBRE:", row["nombre"])
    print("CLASIFICACION:", row["clasificacion"])
    print("PATRONES:", row["patrones_html_detectados"])
    print("=" * 100)
    print(str(row[COLUMNA_TEXTO_CSV])[:3000])
    print()

ID: 496886
NOMBRE: Intimacion
CLASIFICACION: Juicio
PATRONES: ['<p']
la lLoy NO 2 AROSA lo

REMITENTE 772201

/ICIOS DE PROCESAMIENTO SAL ARMANY CAROLINA PERDOMO ORTIZ
Apellido y nombre .

hb razón social q

ISA y E 96.176.316
Rama o al pal ÓN Ne
BISO 2 (1264) Ñ AV. CORRIENTES 2335, OF. 1* “A” Fecha
Código Postal Donnicilio real (1046)
Ñ C.A.B.A. C.A.B.A. Codigo Postai
Provincia Localidad TABA.

Provincia

lidano, remítole copia del requerimiento efecmado 2 CONTACTO GARANTIDO S.A, rigicodo a Ud. las mismas intimaciones y «perdbimientos:
to CD 35094083 O, de fecha 13/12/2025, por resultar sus términos fsos, maliciosos e improcedentes. Ratifico en todos sus términos mis anteriores
que reiteradamente vengo reclamando y que Ud. persiste en 1acumplix, y la cotalidad de los reclamos, jotimaciones y apercibimienzos efecnlados en

estras manifestaciones. Niego y rechazo una vez más encontrarme po reserva de puesto como Ud, refiere, en relación a lo cual le reitero —evamente
Ceabajo a disposici

# Revision de la columna clasificacion

Como ya vimos que existe clasificacion

In [34]:
df_csv["clasificacion"].value_counts(dropna=False)

clasificacion
Embargo               848
Defensa Consumidor    719
Juicio                597
Oficio                562
Mediación             222
SECLO                  38
Name: count, dtype: int64

# 14. Detectar documentos repetidos en CSV

Primero por fila completa:

In [39]:
duplicados_id_csv = df_csv.duplicated(subset=["id"]).sum()
duplicados_texto_csv = df_csv.duplicated(subset=[COLUMNA_TEXTO_CSV]).sum()
duplicados_documento_csv = df_csv.duplicated(
    subset=[COLUMNA_TEXTO_CSV, "nombre", "clasificacion"]
).sum()

print("IDs duplicados:", duplicados_id_csv)
print("Textos legales duplicados:", duplicados_texto_csv)
print("Documentos duplicados por texto + nombre + clasificación:", duplicados_documento_csv)

IDs duplicados: 0
Textos legales duplicados: 738
Documentos duplicados por texto + nombre + clasificación: 714


Después por texto legal, que es más útil:

In [40]:
duplicados_texto_csv = df_csv.duplicated(subset=[COLUMNA_TEXTO_CSV]).sum()

print("Textos legales duplicados:", duplicados_texto_csv)

Textos legales duplicados: 738


### Inspeccion de los duplicados

In [41]:
df_duplicados_texto = df_csv[
    df_csv.duplicated(subset=[COLUMNA_TEXTO_CSV], keep=False)
].sort_values(by=COLUMNA_TEXTO_CSV)

print("Cantidad de filas involucradas en textos duplicados:", df_duplicados_texto.shape[0])

df_duplicados_texto[["id", "nombre", "clasificacion", COLUMNA_TEXTO_CSV]].head(20)

Cantidad de filas involucradas en textos duplicados: 744


,id,nombre,clasificacion,texto_ocr
1860,527273,Embargo - usuario,Embargo,\n
879,508914,Notificacion procesal,Juicio,\n
880,508915,Embargo - usuario,Embargo,\n
882,508917,Embargo - usuario,Embargo,\n
2205,534544,Embargo - usuario,Embargo,\n
2134,532572,Intimacion,Juicio,Error al descargar el archivo desde el bucket.
2096,532534,Embargo - usuario,Embargo,Error al descargar el archivo desde el bucket.
2097,532535,Demanda,Juicio,Error al descargar el archivo desde el bucket.
2103,532541,SECLO,SECLO,Error al descargar el archivo desde el bucket.
2104,532542,SECLO,SECLO,Error al descargar el archivo desde el bucket.


# 15. Detectar posibles columnas de tipo de documento

Vamos a buscar columnas que tengan pocos valores únicos, porque podrían ser tipo, categoría, clasificación, estado, etc.

In [49]:
def columnas_categoricas(df, max_unicos=30):
    resultados = []

    for col in df.columns:
        serie = df[col]

        # Si la columna contiene listas/dicts/sets, la salteamos
        tiene_valores_no_hashables = serie.dropna().apply(
            lambda x: isinstance(x, (list, dict, set))
        ).any()

        if tiene_valores_no_hashables:
            continue

        unicos = serie.nunique(dropna=True)

        if unicos <= max_unicos:
            resultados.append({
                "columna": col,
                "valores_unicos": int(unicos),
                "nulos": int(serie.isna().sum())
            })

    if len(resultados) == 0:
        return pd.DataFrame(columns=["columna", "valores_unicos", "nulos"])

    return pd.DataFrame(resultados).sort_values(by="valores_unicos")

In [50]:
categoricas_csv = columnas_categoricas(df_csv, max_unicos=30)

display(categoricas_csv)

,columna,valores_unicos,nulos
2,tiene_html_detectado,2,0
3,estado_texto_ocr,4,0
1,clasificacion,6,0
0,nombre,18,0


# Ahora detectemos textos inválidos o errores OCR
Esto es clave porque ya vimos:

\n
Error al descargar el archivo desde el bucket.

In [46]:
def detectar_problema_texto_ocr(texto):
    if pd.isna(texto):
        return "nulo"

    texto_str = str(texto).strip()

    if texto_str == "":
        return "vacio"

    if texto_str == "\\n" or texto_str == "\n":
        return "solo_salto_linea"

    if "Error al descargar el archivo desde el bucket" in texto_str:
        return "error_descarga_bucket"

    if len(texto_str) < 50:
        return "texto_muy_corto"

    return "ok"

In [47]:
df_csv["estado_texto_ocr"] = df_csv["texto_ocr"].apply(detectar_problema_texto_ocr)

df_csv["estado_texto_ocr"].value_counts()

estado_texto_ocr
ok                       2248
error_descarga_bucket     696
texto_muy_corto            37
vacio                       5
Name: count, dtype: int64

In [48]:
pd.crosstab(df_csv["clasificacion"], df_csv["estado_texto_ocr"])

estado_texto_ocr,error_descarga_bucket,ok,texto_muy_corto,vacio
clasificacion,,,,
Defensa Consumidor,151,559,9,0
Embargo,209,627,8,4
Juicio,135,455,6,1
Mediación,67,153,2,0
Oficio,122,428,12,0
SECLO,12,26,0,0


# Qué faltaría explorar antes de cerrar CSV
A. Distribución de nombre

In [51]:
df_csv["nombre"].value_counts(dropna=False)

nombre
Embargo - usuario                                    843
Talonarios de Seguimiento                            619
Intimacion                                           420
Solicitud de informacion de usuarios                 340
Mediacion                                            222
Audiencia                                            203
Demanda                                               89
Notificacion procesal                                 88
Reclamo                                               53
SECLO                                                 38
Carta Simple                                          26
Solicitud relacionada con acceso a la informacion     17
Requerimiento                                         10
Propiedad Intelectual                                  5
Embargo - Empleado                                     5
Imputacion administrativa                              4
Allanamiento                                           3
Solicitud de información

### Esto sirve para entender cómo se relacionan nombre y clasificacion.

In [52]:
pd.crosstab(df_csv["clasificacion"], df_csv["nombre"])

nombre,Allanamiento,Audiencia,Carta Simple,Demanda,Embargo - Empleado,Embargo - usuario,Imputacion administrativa,Intimacion,Mediacion,Notificacion procesal,Propiedad Intelectual,Reclamo,Requerimiento,SECLO,Solicitud de informacion de usuarios,Solicitud de información de empleado,Solicitud relacionada con acceso a la informacion,Talonarios de Seguimiento
clasificacion,,,,,,,,,,,,,,,,,,
Defensa Consumidor,0,0,26,0,0,0,4,0,0,0,0,53,0,0,0,0,17,619
Embargo,0,0,0,0,5,843,0,0,0,0,0,0,0,0,0,0,0,0
Juicio,0,0,0,89,0,0,0,420,0,88,0,0,0,0,0,0,0,0
Mediación,0,0,0,0,0,0,0,0,222,0,0,0,0,0,0,0,0,0
Oficio,3,203,0,0,0,0,0,0,0,0,5,0,10,0,340,1,0,0
SECLO,0,0,0,0,0,0,0,0,0,0,0,0,0,38,0,0,0,0


# B. Largo de textos válidos

In [53]:
df_csv["largo_texto_ocr"] = df_csv["texto_ocr"].astype(str).str.strip().str.len()

df_csv.groupby("estado_texto_ocr")["largo_texto_ocr"].describe()

,count,mean,std,min,25%,50%,75%,max
estado_texto_ocr,,,,,,,,
error_descarga_bucket,696.0,46.000000,0.000000,46.0,46.00,46.0,46.00,46.0
ok,2248.0,4349.814947,8393.469502,66.0,1591.75,2938.5,4390.25,117896.0
texto_muy_corto,37.0,27.000000,0.000000,27.0,27.00,27.0,27.00,27.0
vacio,5.0,0.000000,0.000000,0.0,0.00,0.0,0.00,0.0


#### Para los validos
Esto te va a decir si los oficios y embargos tienen textos razonables o si algunos son extremadamente largos/cortos.

In [55]:
df_csv[df_csv["estado_texto_ocr"] == "ok"].groupby("clasificacion")["largo_texto_ocr"].describe()

,count,mean,std,min,25%,50%,75%,max
clasificacion,,,,,,,,
Defensa Consumidor,559.0,2435.345259,6748.705793,104.0,371.50,489.0,2036.00,98817.0
Embargo,627.0,3518.559809,1593.149253,627.0,2398.00,3305.0,4309.50,10929.0
Juicio,455.0,5941.237363,14277.504419,211.0,1731.00,2735.0,4144.50,117896.0
Mediación,153.0,5254.183007,2240.505812,332.0,3726.00,4816.0,6590.00,13170.0
Oficio,428.0,6256.609813,8708.002561,66.0,2975.75,4025.5,5583.50,109110.0
SECLO,26.0,996.423077,287.627561,419.0,794.75,1072.0,1143.25,1417.0


# C. Crear dataset objetivo
Esto ya te deja preparado el dataset real de trabajo.  
textos OCR válidos + clasificación Oficio o Embargo

In [56]:
df_csv_valido = df_csv[df_csv["estado_texto_ocr"] == "ok"].copy()

df_objetivo_csv = df_csv_valido[
    df_csv_valido["clasificacion"].isin(["Oficio", "Embargo"])
].copy()

print("Registros CSV originales:", df_csv.shape[0])
print("Registros válidos:", df_csv_valido.shape[0])
print("Oficios y embargos válidos:", df_objetivo_csv.shape[0])

df_objetivo_csv["clasificacion"].value_counts()

Registros CSV originales: 2986
Registros válidos: 2248
Oficios y embargos válidos: 1055


clasificacion
Embargo    627
Oficio     428
Name: count, dtype: int64

# Análisis final
## Análisis exploratorio del CSV

El archivo CSV contiene 2986 registros y 4 columnas originales: `id`, `texto_ocr`, `nombre` y `clasificacion`.

La columna principal para procesamiento es `texto_ocr`, ya que contiene el texto extraído por OCR de los documentos legales. La columna `clasificacion` permite identificar la categoría general del documento.

No se encontraron valores nulos en las columnas originales, por lo que todos los registros tienen identificador, texto OCR, nombre y clasificación.

La distribución por clasificación muestra que la base contiene más tipos de documentos además de oficios y embargos:

- Embargo: 848
- Defensa Consumidor: 719
- Juicio: 597
- Oficio: 562
- Mediación: 222
- SECLO: 38

La detección inicial de HTML encontró muy pocos casos posibles, por lo que el problema principal del CSV no parece ser HTML, sino ruido propio del OCR: caracteres mal reconocidos, palabras deformadas, saltos de línea, textos incompletos y errores técnicos.

Se detectaron 738 textos legales duplicados. Al inspeccionar ejemplos, muchos corresponden a textos vacíos o mensajes como `Error al descargar el archivo desde el bucket.`, por lo que no deben interpretarse automáticamente como documentos legales repetidos.

Se creó una clasificación auxiliar del estado del texto OCR:

- ok: 2248 registros
- error_descarga_bucket: 696 registros
- texto_muy_corto: 37 registros
- vacio: 5 registros

Para el objetivo inicial de la pasantía, los documentos más relevantes son `Oficio` y `Embargo`. Luego de filtrar solo textos OCR válidos, quedan:

- Embargo: 627 registros válidos
- Oficio: 428 registros válidos

Por lo tanto, el dataset objetivo inicial para limpieza y procesamiento contiene 1055 registros válidos de oficios y embargos.

# Etapa 2 — Lectura guiada de oficios y embargos
Objetivo: Analizar ejemplos reales de Oficio y Embargo para definir qué información importante aparece en cada tipo de documento.

# Crear muestras para leer

In [57]:
# Muestras reproducibles para lectura manual
muestra_oficios = df_objetivo_csv[
    df_objetivo_csv["clasificacion"] == "Oficio"
].sample(n=10, random_state=42)

muestra_embargos = df_objetivo_csv[
    df_objetivo_csv["clasificacion"] == "Embargo"
].sample(n=10, random_state=42)

print("Muestra oficios:", muestra_oficios.shape)
print("Muestra embargos:", muestra_embargos.shape)

Muestra oficios: (10, 8)
Muestra embargos: (10, 8)


# Mostrar ejemplos de oficios

In [58]:
for i, row in muestra_oficios.iterrows():
    print("=" * 100)
    print("TIPO: OFICIO")
    print("ID:", row["id"])
    print("NOMBRE:", row["nombre"])
    print("CLASIFICACION:", row["clasificacion"])
    print("=" * 100)
    print(str(row["texto_ocr"])[:4000])
    print()

TIPO: OFICIO
ID: 546322
NOMBRE: Solicitud de informacion de usuarios
CLASIFICACION: Oficio
236
pola 13

OFICIO JUDICIAL
Buenos Aires, de marzo de 2026

MERCADOLIBRE SRL
Av. Caseros 3039, Piso 2 —- CABA.
Ss J D

Tengo el agrado de dirigirme a Ud. en mi carácter de letrado de la parte actora en el
expediente caratulado “DE LOS SANTOS, PABLO ROBERTO C/ GIMENEZ, PABLO
FABIAN Y OTROS S/DAÑOS Y PERJUICIOS(ACC.TRAN. C/LES. O MUERTE)” (Expte.
43264/2021) que tramita en el Juzgado Nacional en lo Civil N*65, sito en Inmigrantes 1950
piso 6?, CABA, a cargo de la Dra. María Gabriela FERNANDEZ ZURITA, Secretaría única a
cargo del Dr. Diego Martín DE LA IGLESIA. Se ha dispuesto a dirigir a Ud. el presente a fin
de solicitarle que informe el último domicilio registrado del señor Cairo CLAROS PATZI DNI
93.946.731.

El auto que ordena la medida dice así en su parte pertinente: “Buenos Aires, 16 de
diciembre de 2025 (...) Sin perjuicio de destacar que ya ha sido ordenada la publicación de
edictos, líbre

# Mostrar ejemplos de embargos

In [59]:
for i, row in muestra_embargos.iterrows():
    print("=" * 100)
    print("TIPO: EMBARGO")
    print("ID:", row["id"])
    print("NOMBRE:", row["nombre"])
    print("CLASIFICACION:", row["clasificacion"])
    print("=" * 100)
    print(str(row["texto_ocr"])[:4000])
    print()

TIPO: EMBARGO
ID: 544832
NOMBRE: Embargo - usuario
CLASIFICACION: Embargo
OFICIO JUDICIAL

/ L/)

Buenos Aires, de Febrero/de/2026
Al Sr. Apoderado de |
“MERCADO LIBRE S.R.L.”
S / D

De mi consideración:

Tengo el agrado de dirigirme a Ud. en los autos
caratulados “CHUQUISACA, SERGIO HORACIO C/ MENUDENCIAS
S.A, Y OTROS S/ DESPIDO ", Expediente. Nro. 21903/14, que tramita
por ante el Juzgado Nacional de Primera Instancia del Trabajo Nro. 35, sito
en la calle Tte. Gral. Juan D. Perón 990 Piso 7mo. de C.A.B.A., a mi cargo,
Secretaría única a cargo del DR. MARIANO GILLETTE TORRILLA,
para informarle que se ha decretado embargo sobre bienes de la demandada
“FAENADORA ARGENTINA S.A.” CUIT 33-71237566-9 hasta cubrir la
suma de $ 31.177,68.- (en concepto de honorarios del perito contador), con
más la de $9.000.- presupuestados provisoriamente para responder a
IMÍOreSES Y COSÍAS occ

Se le hace saber que deberá retener de las sumas que tenga la embargada
depositadas en esa entidad, por cualquier

# Crear una plantilla de análisis manual

La idea es leer cada caso y llenar una tabla con observaciones.

In [60]:
columnas_analisis = [
    "id",
    "clasificacion",
    "nombre",
    "organismo_o_juzgado",
    "destinatario",
    "persona_principal",
    "dni_cuit_cuil",
    "expediente",
    "caratula",
    "fecha",
    "monto",
    "cuenta_cbu_cvu_alias",
    "datos_solicitados",
    "email_respuesta",
    "observaciones",
    "calidad_ocr"
]

analisis_manual = pd.DataFrame(columns=columnas_analisis)

analisis_manual

,id,clasificacion,nombre,organismo_o_juzgado,destinatario,persona_principal,dni_cuit_cuil,expediente,caratula,fecha,monto,cuenta_cbu_cvu_alias,datos_solicitados,email_respuesta,observaciones,calidad_ocr


# Crear archivo de muestras para revisar más cómodo
Esto te genera archivos .txt separados para leerlos en VS Code sin perderte.

In [63]:
SAMPLES_DIR = PROCESSED_DIR / "muestras_lectura"
SAMPLES_DIR.mkdir(parents=True, exist_ok=True)

def guardar_muestras_lectura(df, tipo, cantidad=10):
    muestra = df[df["clasificacion"] == tipo].sample(n=cantidad, random_state=42)

    for _, row in muestra.iterrows():
        filename = f"{tipo.lower()}_{row['id']}.txt"
        path = SAMPLES_DIR / filename

        contenido = f"""TIPO: {tipo}
ID: {row['id']}
NOMBRE: {row['nombre']}
CLASIFICACION: {row['clasificacion']}

{'=' * 80}

{row['texto_ocr']}
"""

        with open(path, "w", encoding="utf-8") as f:
            f.write(contenido)

    print(f"Muestras guardadas para {tipo} en:", SAMPLES_DIR)

In [64]:
guardar_muestras_lectura(df_objetivo_csv, "Oficio", cantidad=10)
guardar_muestras_lectura(df_objetivo_csv, "Embargo", cantidad=10)

Muestras guardadas para Oficio en: E:\CAROLA\CAROLA TRABAJO\PASANTIA UNNE - AGENTE ASUNTOS LEGALES\pasantia-segem-agentes-ia\data\processed\muestras_lectura
Muestras guardadas para Embargo en: E:\CAROLA\CAROLA TRABAJO\PASANTIA UNNE - AGENTE ASUNTOS LEGALES\pasantia-segem-agentes-ia\data\processed\muestras_lectura


# Crear una tabla manual inicial

In [65]:
plantilla_path = PROCESSED_DIR / "plantilla_analisis_manual_oficios_embargos.csv"

analisis_manual.to_csv(plantilla_path, index=False, encoding="utf-8-sig")

print("Plantilla guardada en:", plantilla_path)

Plantilla guardada en: E:\CAROLA\CAROLA TRABAJO\PASANTIA UNNE - AGENTE ASUNTOS LEGALES\pasantia-segem-agentes-ia\data\processed\plantilla_analisis_manual_oficios_embargos.csv


# Etapa 3 - Limpieza de textos OCR

Objetivo: limpiar los textos de oficios y embargos válidos para dejarlos en un formato más usable para análisis posterior con IA.

La limpieza incluye:

- corrección básica de encoding y acentos rotos;
- limpieza preventiva de HTML con Beautiful Soup;
- normalización de espacios;
- normalización de tabulaciones;
- normalización de saltos de línea;
- reducción de ruido visual;
- conversión a Markdown;
- conservación del texto original.

In [ ]:
import sys
from pathlib import Path

# Permite importar src tanto al ejecutar desde notebooks/ como desde la raiz.
directorio_actual = Path.cwd().resolve()
if directorio_actual.name == "notebooks":
    PROJECT_ROOT = directorio_actual.parent
elif (directorio_actual / "src").exists():
    PROJECT_ROOT = directorio_actual

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

from src.cleaning import (
    convertir_a_markdown,
    guardar_muestras_markdown,
    limpiar_texto_legal,
    normalizar_para_busqueda,
)

In [ ]:
# Reconstruccion defensiva para ejecuciones parciales del notebook.
if "df_objetivo_csv" not in globals():
    columnas_requeridas = {"estado_texto_ocr", "clasificacion"}
    faltantes = columnas_requeridas.difference(df_csv.columns)
    if faltantes:
        raise KeyError(
            "No se puede reconstruir df_objetivo_csv. "
            f"Faltan columnas en df_csv: {sorted(faltantes)}"
        )

    df_objetivo_csv = df_csv[
        (df_csv["estado_texto_ocr"] == "ok")
        & (df_csv["clasificacion"].isin(["Oficio", "Embargo"]))
    ].copy()

df_limpieza = df_objetivo_csv.copy()
print("Registros a limpiar:", len(df_limpieza))

In [ ]:
# Se conserva texto_ocr y todas las columnas originales sin sobrescribirlas.
df_limpieza["largo_original"] = (
    df_limpieza["texto_ocr"].fillna("").astype(str).str.len()
)
df_limpieza["texto_limpio"] = df_limpieza["texto_ocr"].apply(
    limpiar_texto_legal
)
df_limpieza["texto_normalizado_busqueda"] = df_limpieza[
    "texto_limpio"
].apply(normalizar_para_busqueda)
df_limpieza["largo_limpio"] = df_limpieza["texto_limpio"].str.len()
df_limpieza["texto_markdown"] = df_limpieza.apply(
    convertir_a_markdown,
    axis=1,
)
df_limpieza["diferencia_largo"] = (
    df_limpieza["largo_original"] - df_limpieza["largo_limpio"]
)

df_limpieza.head()

## Comparación de calidad

In [ ]:
df_limpieza[
    ["largo_original", "largo_limpio", "diferencia_largo"]
].describe()

In [ ]:
ids_revision = [504422, 498355]
mascara_ids = df_limpieza["id"].astype(str).isin(
    [str(id_revision) for id_revision in ids_revision]
)
ejemplos_revision = df_limpieza[mascara_ids]

if len(ejemplos_revision) != len(ids_revision):
    muestras = []
    for tipo in ["Oficio", "Embargo"]:
        candidatos = df_limpieza[df_limpieza["clasificacion"] == tipo]
        if not candidatos.empty:
            muestras.append(candidatos.sample(n=1, random_state=42))
    ejemplos_revision = pd.concat(muestras) if muestras else df_limpieza.head(0)

for _, row in ejemplos_revision.iterrows():
    print("=" * 100)
    print(f"ID: {row['id']} | CLASIFICACION: {row['clasificacion']}")
    print("-" * 100)
    print("ANTES (texto_ocr):")
    print(str(row["texto_ocr"])[:4000])
    print("-" * 100)
    print("DESPUES (texto_limpio):")
    print(row["texto_limpio"][:4000])
    print()

## Guardar resultados procesados

In [ ]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
MARKDOWN_DIR = PROCESSED_DIR / "muestras_markdown"

guardar_muestras_markdown(
    df_limpieza,
    MARKDOWN_DIR,
    cantidad_por_tipo=5,
    random_state=42,
)

csv_limpio_path = PROCESSED_DIR / "oficios_embargos_limpios.csv"
json_limpio_path = PROCESSED_DIR / "oficios_embargos_limpios.json"

df_limpieza.to_csv(csv_limpio_path, index=False, encoding="utf-8-sig")
df_limpieza.to_json(
    json_limpio_path,
    orient="records",
    force_ascii=False,
    indent=2,
)

print("Muestras Markdown:", MARKDOWN_DIR)
print("CSV limpio:", csv_limpio_path)
print("JSON limpio:", json_limpio_path)

## Criterio de limpieza

La limpieza aplicada es **conservadora**: no convierte el texto principal a minúsculas, no elimina acentos y no intenta corregir ni descartar palabras propias del OCR que podrían contener información legal relevante.

Beautiful Soup se utiliza de manera preventiva para retirar etiquetas, scripts, estilos y entidades HTML si aparecen. La exploración inicial mostró poco HTML real; el problema principal observado fue el ruido propio del OCR.

La columna original `texto_ocr` se conserva sin modificaciones. Se agregan las columnas:

- `texto_limpio`
- `texto_normalizado_busqueda`
- `texto_markdown`
- `largo_original`
- `largo_limpio`
- `diferencia_largo`

# VERIFICACION 

In [67]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

csv_limpio_path = PROCESSED_DIR / "oficios_embargos_limpios.csv"

df_limpieza = pd.read_csv(csv_limpio_path)

print("Archivo existe:", csv_limpio_path.exists())
print("Shape df_limpieza:", df_limpieza.shape)

Archivo existe: True
Shape df_limpieza: (1055, 14)


In [68]:
columnas_esperadas = [
    "texto_ocr",
    "texto_limpio",
    "texto_normalizado_busqueda",
    "texto_markdown",
    "largo_original",
    "largo_limpio",
    "diferencia_largo"
]

for col in columnas_esperadas:
    print(col, "OK" if col in df_limpieza.columns else "FALTA")

texto_ocr OK
texto_limpio OK
texto_normalizado_busqueda OK
texto_markdown OK
largo_original OK
largo_limpio OK
diferencia_largo OK


In [69]:
print("Nulos en texto_limpio:", df_limpieza["texto_limpio"].isna().sum())
print("Vacíos en texto_limpio:", (df_limpieza["texto_limpio"].astype(str).str.strip() == "").sum())

Nulos en texto_limpio: 0
Vacíos en texto_limpio: 0


In [70]:
df_limpieza[["largo_original", "largo_limpio", "diferencia_largo"]].describe()

,largo_original,largo_limpio,diferencia_largo
count,1055.000000,1055.000000,1055.000000
mean,4631.088152,4516.222749,114.865403
std,5834.164015,5379.115874,849.441643
min,67.000000,65.000000,1.000000
25%,2598.000000,2586.000000,2.000000
50%,3586.000000,3575.000000,3.000000
75%,4649.000000,4631.000000,4.000000
max,109112.000000,105529.000000,15023.000000


In [71]:
for tipo in ["Oficio", "Embargo"]:
    row = df_limpieza[df_limpieza["clasificacion"] == tipo].sample(1, random_state=42).iloc[0]

    print("=" * 100)
    print("TIPO:", tipo)
    print("ID:", row["id"])
    print("=" * 100)

    print("\n--- ORIGINAL ---\n")
    print(row["texto_ocr"][:1500])

    print("\n--- LIMPIO ---\n")
    print(row["texto_limpio"][:1500])

    print("\n--- MARKDOWN ---\n")
    print(row["texto_markdown"][:1500])

TIPO: Oficio
ID: 546322

--- ORIGINAL ---

236
pola 13

OFICIO JUDICIAL
Buenos Aires, de marzo de 2026

MERCADOLIBRE SRL
Av. Caseros 3039, Piso 2 —- CABA.
Ss J D

Tengo el agrado de dirigirme a Ud. en mi carácter de letrado de la parte actora en el
expediente caratulado “DE LOS SANTOS, PABLO ROBERTO C/ GIMENEZ, PABLO
FABIAN Y OTROS S/DAÑOS Y PERJUICIOS(ACC.TRAN. C/LES. O MUERTE)” (Expte.
43264/2021) que tramita en el Juzgado Nacional en lo Civil N*65, sito en Inmigrantes 1950
piso 6?, CABA, a cargo de la Dra. María Gabriela FERNANDEZ ZURITA, Secretaría única a
cargo del Dr. Diego Martín DE LA IGLESIA. Se ha dispuesto a dirigir a Ud. el presente a fin
de solicitarle que informe el último domicilio registrado del señor Cairo CLAROS PATZI DNI
93.946.731.

El auto que ordena la medida dice así en su parte pertinente: “Buenos Aires, 16 de
diciembre de 2025 (...) Sin perjuicio de destacar que ya ha sido ordenada la publicación de
edictos, líbrense los oficios que se solicitan en los términos

## Conclusión de la transformación a Markdown

La etapa de limpieza procesó 1055 documentos válidos clasificados como `Oficio` o `Embargo`.

La limpieza fue conservadora: se mantuvo el contenido legal original lo más intacto posible, evitando transformaciones agresivas como pasar todo a minúsculas, eliminar acentos o corregir automáticamente errores OCR.

La columna `texto_ocr` se conservó como fuente original, mientras que se generaron versiones auxiliares:

- `texto_limpio`: texto normalizado y más legible.
- `texto_normalizado_busqueda`: versión auxiliar para búsquedas.
- `texto_markdown`: versión estructurada para uso posterior con modelos de IA.

La conversión a Markdown permite separar metadatos del contenido legal y deja cada documento en un formato más claro para tareas futuras de extracción de entidades, comparación entre oficio/respuesta y revisión automática.

La validación mostró que no hay textos limpios nulos ni vacíos. La mayoría de los documentos tuvo una diferencia mínima entre el largo original y el largo limpio, lo que indica que la limpieza no fue destructiva.

# Etapa 4 - Separaci?n del dataset de embargos

En esta etapa se separa el dataset de embargos para trabajar espec?ficamente con esos documentos en an?lisis posteriores. Se parte de la base limpia `oficios_embargos_limpios.csv`, se filtran ?nicamente los registros con `clasificacion == "Embargo"` y se guarda un nuevo archivo sin modificar la base limpia original.


In [1]:
from pathlib import Path
import pandas as pd

directorio_actual = Path.cwd().resolve()
if directorio_actual.name == "notebooks":
    PROJECT_ROOT = directorio_actual.parent
elif (directorio_actual / "data" / "processed").exists():
    PROJECT_ROOT = directorio_actual
else:
    PROJECT_ROOT = directorio_actual.parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

archivo_limpio = PROCESSED_DIR / "oficios_embargos_limpios.csv"
archivo_embargos = PROCESSED_DIR / "embargos_limpios.csv"

df_limpio = pd.read_csv(archivo_limpio)

if "clasificacion" not in df_limpio.columns:
    raise ValueError("No existe la columna 'clasificacion' en el archivo limpio.")

df_embargos = df_limpio[df_limpio["clasificacion"] == "Embargo"].copy()

df_embargos.to_csv(archivo_embargos, index=False, encoding="utf-8-sig")

print("Archivo de embargos guardado en:", archivo_embargos)
print("Existe archivo:", archivo_embargos.exists())
print("Cantidad de embargos:", df_embargos.shape[0])
print(df_embargos["clasificacion"].value_counts())


Archivo de embargos guardado en: E:\CAROLA\CAROLA TRABAJO\PASANTIA UNNE - AGENTE ASUNTOS LEGALES\pasantia-segem-agentes-ia\data\processed\embargos_limpios.csv
Existe archivo: True
Cantidad de embargos: 627
clasificacion
Embargo    627
Name: count, dtype: int64
